# Experiment 07: SVM Classifiers (Linear vs Nonlinear RBF Kernels)


### 1. Import Libraries


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

print("Libraries imported successfully.")


### 2. Load and Filter Iris Dataset (Setosa vs Versicolor)


In [ ]:
dataset = load_iris(as_frame=True)
iris_df = dataset.frame.copy()

# Keep two Iris classes for binary classification: Setosa (0) and Versicolor (1)
iris_df = iris_df[iris_df["target"].isin([0, 1])].copy()
X = iris_df[dataset.feature_names].copy()
y = iris_df["target"].astype(int)
y.name = "target"

df = X.copy()
df["target"] = y.map({0: "Iris Setosa (0)", 1: "Iris Versicolor (1)"})

print("Dataset shape (samples, features):", X.shape)
print("
First five rows:")
display(df.head())

print("
Class distribution:")
print(y.value_counts().sort_index().rename(index={0: "Iris Setosa (0)", 1: "Iris Versicolor (1)"}))

print("
Missing values in features:", int(X.isnull().sum().sum()))


### 3. Split into Training and Testing Sets (80:20 Stratified)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples :", X_test.shape[0])


### 4. Create and Train Linear & RBF SVM Classifiers


In [ ]:
models = {
    "Linear SVM": Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("svm", SVC(kernel="linear", C=1.0))
    ]),
    "RBF (nonlinear) SVM": Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("svm", SVC(kernel="rbf", C=1.0, gamma="scale"))
    ])
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"{name} trained successfully.")


### 5. Predict and Calculate Evaluation Metrics


In [ ]:
results = []
predictions = {}

for name, model in models.items():
    y_pred = model.predict(X_test)
    predictions[name] = y_pred
    
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision (Versicolor)": precision_score(y_test, y_pred, pos_label=1, zero_division=0),
        "Recall (Versicolor)": recall_score(y_test, y_pred, pos_label=1, zero_division=0),
        "F1-score (Versicolor)": f1_score(y_test, y_pred, pos_label=1, zero_division=0)
    })

results_df = pd.DataFrame(results).set_index("Model")
display(results_df.style.format("{:.4f}"))


### 6. Display Confusion Matrices


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ax, (name, y_pred) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Iris Setosa (0)", "Iris Versicolor (1)"]
    )
    disp.plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)
    ax.set_title(name)

plt.tight_layout()
plt.show()


### 7. Compare Metrics Visually


In [ ]:
metric_columns = [
    "Accuracy", "Precision (Versicolor)", "Recall (Versicolor)", "F1-score (Versicolor)"
]

ax = results_df[metric_columns].plot(kind="bar", figsize=(11, 5), ylim=(0, 1.05))
ax.set_title("Linear vs Nonlinear (RBF) SVM Performance")
ax.set_ylabel("Score")
ax.set_xlabel("Classifier")
ax.grid(axis="y", linestyle="--", alpha=0.4)
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()


### 8. Classification Reports


In [ ]:
for name, y_pred in predictions.items():
    print("=" * 65)
    print(name)
    print(classification_report(
        y_test, y_pred,
        labels=[0, 1],
        target_names=["Iris Setosa (0)", "Iris Versicolor (1)"],
        digits=4,
        zero_division=0
    ))
